In [1]:
import pandas as pd
import numpy as np
import utils.fetch_dbs
import json
import mysql.connector
pd.set_option('display.max_columns', None)

In [2]:
datasets = pd.read_csv('HDCA_v2_datasets.csv')
datasets

,title,doi,accession,institute,assay,key,avialability,status,comment,simone comment,link
0,Calvanese_et_al_2022_nature,https://doi.org/10.1038/s41586-022-04571-x,GSE162950,Broad_Stem_Cell_Research_centre,NaN,NaN,open,reprocessed by Alex,"we use just four samples, I guess AGM stands f...",NaN,NaN
1,Dong_et_al_2022_CancerCell,https://doi.org/10.1016/j.ccell.2020.08.014,GSE137804,Children's Hospital Affiliated to Fudan Univer...,NaN,NaN,open,NaN,"just four samples, to be selected by names",NaN,NaN
2,Braun_et_al._2022_bioRxiv,https://www.science.org/doi/10.1126/science.ad...,EGAD50000001295,Karolinska,NaN,NaN,EGA,reprocessed by Alex,EGAS00001004107,NaN,NaN
3,Braun_et_al._2022_bioRxiv,https://www.science.org/doi/10.1126/science.ad...,EGAD00001006049,Karolinska,NaN,NaN,EGA,reprocessed by Alex,EGAS00001004107,NaN,NaN
4,Colin_et_al_2022_ScienceDirect,https://doi.org/10.1016/j.jtos.2021.03.010,GSE155683,Newcastle_University,NaN,NaN,open,NaN,NaN,NaN,NaN
5,Yu_et_al_2020_Cell,https://doi.org/10.1016/j.cell.2021.04.028,E-MTAB-10187,Roche,NaN,NaN,open,reprocessed by Alex,NaN,NaN,NaN
6,Miller_et_al_2020_Developmental_Cell,https://doi.org/10.1016/j.devcel.2020.01.033,E-MTAB-8221,University_of_Michigan,NaN,NaN,open,reprocessed by Alex,NaN,NaN,NaN
7,Sridhar_et_al_2020_CellPress,https://doi.org/10.1016/j.celrep.2020.01.007,GSE142526,University_of_Washington,NaN,NaN,open,NaN,NaN,NaN,NaN
8,Garcia_Alonso_et_al_2022_nature,https://doi.org/10.1038/s41586-022-04918-4,E-MTAB-10551,Wellcome_Sanger,10x RNA-seq,NaN,open,reprocessed by Alex,NaN,NaN,NaN
9,Garcia_Alonso_et_al_2022_nature,https://doi.org/10.1038/s41586-022-04918-4,E-MTAB-11708,Wellcome_Sanger,10x multiome,NaN,open,NaN,not sure it was used,NaN,NaN


In [3]:
datasets.accession.value_counts()

accession
GSE162950                           1
GSE137804                           1
E-MTAB-11911                        1
E-MTAB-8813                         1
GSE260715                           1
E-MTAB-6701                         1
E-MTAB-14385                        1
whole_embryo_11pcw                  1
GSE157329                           1
HRA005835                           1
GSE264407                           1
HRA002757                           1
E-MTAB-7407                         1
E-MTAB-13071                        1
E-MTAB-10552                        1
E-MTAB-11673                        1
EGAD00001015384                     1
PRJEB77091                          1
GSE271304                           1
E-MTAB-11520                        1
E-MTAB-11504                        1
E-MTAB-11343                        1
E-MTAB-11708                        1
EGAD50000001295                     1
EGAD00001006049                     1
GSE155683                           1
E-

# Collect metadata from databases
## ArrayExpress

In [ ]:
inx = datasets.accession.str.startswith('E-MTAB')
inx = inx[inx].index
for i in inx:
    a = datasets.accession[i]
    print(a)
    url = 'https://www.ebi.ac.uk/biostudies/files/'+a+'/'+a+'.sdrf.txt'
    if pd.notna(datasets.key[i]):
        url = url+"?key="+datasets.key[i]
    d = pd.read_csv(url, sep="\t")
    d.to_csv("datasets_metadata/"+a+".csv")

## EGA

In [ ]:
f = datasets.accession.str.startswith('EGAD')
for a in datasets.accession[f]:
    print(a)
    d = utils.fetch_dbs.get_EGAD_files(a)
    d.to_csv("datasets_metadata/"+a+".csv")

## GEO

In [ ]:
f = datasets.accession.str.startswith('GSE')
for a in datasets.accession[f]:
    print(a)
    url = 'https://ftp.ncbi.nlm.nih.gov/geo/series/'+a[:6]+'nnn/'+a+'/soft/'+a+'_family.soft.gz'
    d = utils.fetch_dbs.get_geo_family_soft_samples(url)
    d.to_csv("datasets_metadata/"+a+".csv")

## ENA

In [ ]:
f = datasets.accession.str.startswith('PRJEB')
for a in datasets.accession[f]:
    print(a)
    url = 'https://www.ebi.ac.uk/ena/portal/api/filereport?accession='+a+'&result=read_run&fields=study_accession,experiment_accession,sample_accession,secondary_sample_accession,sample_title&format=tsv'
    d = pd.read_csv(url, sep="\t")
    d.to_csv("datasets_metadata/"+a+".csv")

## CNCB
didn't find working API, anyway we need to get access first

# Concatenate metadata
The goal is to get list of samples with minimal info:

1. dataset_acc - primary id
2. sample_acc - primary id
3. author_sample_id - the name used in object (hopefully)
4. irods_name - how we identify it locally
5. [library_type]

## Array Express

In [168]:
f = datasets.accession.str.startswith('E-MTAB')
dss = []
for a in datasets.accession[f]:
    d = pd.read_csv("datasets_metadata/"+a+".csv")
    d['dataset_acc'] = a
    # hooks for non-usual datasets
    if a == 'E-MTAB-11278':
        d['Source Name'] = d['Derived Array Data File'].str.replace('.tar.gz','')
    if a == 'E-MTAB-8813':
        d['Source Name'] = d['Derived Array Data File.1'].str.replace('.tar.gz','')
    if a == 'E-MTAB-8901':
        d['Source Name'] = [x[0] for x in d['Comment[read1 file]'].str.split('_')]
    if a == 'E-MTAB-8581':
        d['Source Name'] = d['Comment[original source name]']
    dss.append(d)

common_cols = list(set.intersection(*(set(df.columns) for df in dss)))

ae = pd.concat([df[common_cols] for df in dss], ignore_index=True)
ae[:3]

,Unnamed: 0,Source Name,Comment[cell barcode size],Performer,Characteristics[developmental stage],Comment[LIBRARY_SOURCE],Comment[primer],Protocol REF.2,Technology Type,Protocol REF.3,Comment[umi barcode size],Assay Name,dataset_acc,Scan Name,Characteristics[organism],Comment[LIBRARY_SELECTION],Comment[ENA_SAMPLE],Comment[FASTQ_URI],Comment[LIBRARY_LAYOUT],Comment[ENA_EXPERIMENT],Comment[umi barcode read],Comment[cell barcode read],Comment[end bias],Material Type,Comment[umi barcode offset],Protocol REF,Comment[library construction],Comment[BioSD_SAMPLE],Comment[ENA_RUN],Comment[LIBRARY_STRATEGY],Characteristics[individual],Comment[single cell isolation],Extract Name,Protocol REF.1,Comment[cell barcode offset],Comment[input molecule]
0,0,H9-Definitive-endoderm,16.0,University of Michigan Advanced Genomics Core,not available,TRANSCRIPTOMIC SINGLE CELL,oligo-dT,P-MTAB-107112,sequencing assay,P-MTAB-107113,10.0,H9-DE_S1_L001,E-MTAB-10187,H9-DE_S1_L001.bam,Homo sapiens,Oligo-dT,ERS5890137,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,SINGLE,ERX5202382,read 1,read 1,3 prime tag,organoid,16.0,P-MTAB-107118,10xV2,SAMEA8203536,ERR5417814,RNA-Seq,10,10x,H9-Definitive-endoderm,P-MTAB-107111,0.0,polyA RNA
1,1,H9-Definitive-endoderm,16.0,University of Michigan Advanced Genomics Core,not available,TRANSCRIPTOMIC SINGLE CELL,oligo-dT,P-MTAB-107112,sequencing assay,P-MTAB-107113,10.0,H9-DE_S1_L002,E-MTAB-10187,H9-DE_S1_L002.bam,Homo sapiens,Oligo-dT,ERS5890137,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,SINGLE,ERX5202382,read 1,read 1,3 prime tag,organoid,16.0,P-MTAB-107118,10xV2,SAMEA8203536,ERR5417815,RNA-Seq,10,10x,H9-Definitive-endoderm,P-MTAB-107111,0.0,polyA RNA
2,2,H9-Definitive-endoderm,16.0,University of Michigan Advanced Genomics Core,not available,TRANSCRIPTOMIC SINGLE CELL,oligo-dT,P-MTAB-107112,sequencing assay,P-MTAB-107113,10.0,H9-DE_S1_L003,E-MTAB-10187,H9-DE_S1_L003.bam,Homo sapiens,Oligo-dT,ERS5890137,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,SINGLE,ERX5202382,read 1,read 1,3 prime tag,organoid,16.0,P-MTAB-107118,10xV2,SAMEA8203536,ERR5417816,RNA-Seq,10,10x,H9-Definitive-endoderm,P-MTAB-107111,0.0,polyA RNA


### Remove unwanted samples

In [169]:
ae.loc[ae['Comment[ENA_RUN]'].isna(),'dataset_acc'].value_counts() # samples with no raw data on ArrayExpress, remove them

dataset_acc
E-MTAB-10551    28
E-MTAB-11708     4
Name: count, dtype: int64

In [170]:
ae = ae[~ae['Comment[ENA_RUN]'].isna()]

In [171]:
ae['Comment[LIBRARY_STRATEGY]'].value_counts()

Comment[LIBRARY_STRATEGY]
RNA-Seq     1610
OTHER        347
ATAC-seq     216
Name: count, dtype: int64

In [172]:
ae.loc[ae['Comment[LIBRARY_STRATEGY]']=='ATAC-seq','Comment[library construction]'].value_counts()

Comment[library construction]
scATAC-seq    180
other          36
Name: count, dtype: int64

In [173]:
ae.loc[ae['Comment[LIBRARY_STRATEGY]']=='RNA-Seq','Comment[library construction]'].value_counts()

Comment[library construction]
10xV2        409
10x 5' v2    384
10x 5' v1    287
10xV3        153
10x 3' v3    144
10xV1        140
10x 3' v2     66
10x 3’ v2     27
Name: count, dtype: int64

In [174]:
ae.loc[ae['Comment[LIBRARY_STRATEGY]']=='OTHER','Comment[library construction]'].value_counts()

Comment[library construction]
10x 5' v1             116
10x 3' v2              93
10x TCR enrichment     87
10x BCR enrichment     48
10x 3' v3               3
Name: count, dtype: int64

In [175]:
ae = ae.loc[ae['Comment[LIBRARY_STRATEGY]']!='ATAC-seq',]
ae = ae.loc[ae['Comment[library construction]']!='10x TCR enrichment',]
ae = ae.loc[ae['Comment[library construction]']!='10x BCR enrichment',]

In [176]:
ae['Comment[library construction]'].value_counts()

Comment[library construction]
10xV2        409
10x 5' v1    403
10x 5' v2    384
10x 3' v2    159
10xV3        153
10x 3' v3    147
10xV1        140
10x 3’ v2     27
Name: count, dtype: int64

In [177]:
ae['Material Type'].value_counts()

Material Type
cell             1033
organism part     739
organoid           50
Name: count, dtype: int64

In [178]:
ae.loc[ae['Material Type']=='organoid','dataset_acc'].value_counts()

dataset_acc
E-MTAB-10187    32
E-MTAB-8221     18
Name: count, dtype: int64

In [179]:
ae = ae.loc[ae['Material Type']!='organoid',['dataset_acc','Comment[ENA_SAMPLE]','Source Name','Comment[library construction]']].drop_duplicates()

In [180]:
ae

,dataset_acc,Comment[ENA_SAMPLE],Source Name,Comment[library construction]
32,E-MTAB-10187,ERS5890146,HT-185_Duodenum,10xV2
35,E-MTAB-10187,ERS5890147,HT-188_Duodenum,10xV2
41,E-MTAB-10187,ERS5890148,HT-228_Colon,10xV2
49,E-MTAB-10187,ERS5890149,HT-228_Ileum,10xV2
57,E-MTAB-10187,ERS5890150,HT-228_Liver,10xV2
...,...,...,...,...
2200,E-MTAB-8581,ERS4228604,WSSS8062675,10xV2
2201,E-MTAB-8581,ERS4228662,WSSS8084742,10xV3
2202,E-MTAB-8581,ERS4228663,WSSS8084743,10xV3
2203,E-MTAB-8581,ERS4228664,WSSS8084744,10xV3


In [181]:
# keep only GEX Sanger id in author_sample_id for To_et_al_2024_nature
f = ae.dataset_acc=='E-MTAB-14385'
ae.loc[f,'Source Name'] = [x[0] for x in ae.loc[f,'Source Name'].str.split('_and_')]

## EGA

In [182]:
f = datasets.accession.str.startswith('EGAD')
dss = []
for a in datasets.accession[f]:
    d = pd.read_csv("datasets_metadata/"+a+".csv")
    d['dataset_acc'] = a
    dss.append(d)

common_cols = list(set.intersection(*(set(df.columns) for df in dss)))

ega = pd.concat([df[common_cols] for df in dss], ignore_index=True)
ega = ega[['dataset_acc','EGAN','AUTHOR_ID','LIBRARY_TYPE']].drop_duplicates()
ega[:3]

,dataset_acc,EGAN,AUTHOR_ID,LIBRARY_TYPE
0,EGAD50000001295,EGAN50000223075,10X147_1,10Xchromium 3' single cell version 2
1,EGAD50000001295,EGAN50000223066,10X186_1,10Xchromium 3' single cell version 2
2,EGAD50000001295,EGAN50000223076,10X186_4,10Xchromium 3' single cell version 2


In [183]:
pd.crosstab(ega.LIBRARY_TYPE,ega.dataset_acc)

dataset_acc,EGAD00001009386,EGAD00001009801,EGAD00001015384,EGAD50000001295
LIBRARY_TYPE,,,,
10Xchromium 3' single cell version 2,0,0,0,69
Chromium Visium,0,7,7,0
Chromium single cell 3 prime v3,0,50,0,0
Chromium single cell 5 prime,28,0,7,0
Chromium single cell TCR,0,0,5,0


In [184]:
ega = ega[~ega.LIBRARY_TYPE.isin(['Chromium single cell TCR','Chromium Visium'])]
pd.crosstab(ega.LIBRARY_TYPE,ega.dataset_acc)

dataset_acc,EGAD00001009386,EGAD00001009801,EGAD00001015384,EGAD50000001295
LIBRARY_TYPE,,,,
10Xchromium 3' single cell version 2,0,0,0,69
Chromium single cell 3 prime v3,0,50,0,0
Chromium single cell 5 prime,28,0,7,0


## GEO

In [185]:
f = datasets.accession.str.startswith('GSE')
dss = []
for a in datasets.accession[f]:
    d = pd.read_csv("datasets_metadata/"+a+".csv")
    d['dataset_acc'] = a
    dss.append(d)

common_cols = list(set.intersection(*(set(df.columns) for df in dss)))

geo = pd.concat([df[common_cols] for df in dss], ignore_index=True)
geo[:3]

,library_selection,Unnamed: 0,data_row_count,supplementary_file_1,library_strategy,contact_name,status,title,series_id,type,taxid_ch1,instrument_model,sample_id,contact_country,organism_ch1,channel_count,contact_address,molecule_ch1,characteristics_ch1,dataset_acc,contact_zip/postal_code,submission_date,extract_protocol_ch1,contact_institute,contact_city,last_update_date,platform_id,source_name_ch1,library_source,geo_accession,data_processing,relation
0,cDNA,0,0,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4968...,RNA-Seq,"Feiyang,,Ma",Public on Jan 20 2022,aorta-gonad-mesonephros 4.5 week,GSE162950,SRA,9606,Illumina NovaSeq 6000,GSM4968831,USA,Homo sapiens,1,"610, Charles Young Dr East, TLSB 3000C",total RNA,time: 4wk,GSE162950,90095,Dec 09 2020,"For single cell RNA-sequencing, dissociated ce...",UCLA,Los Angeles,Jan 20 2022,GPL24676,aorta-gonad-mesonephros,transcriptomic,GSM4968831,Sequencing was performed on Illumina NovaSeq 6...,BioSample: https://www.ncbi.nlm.nih.gov/biosam...
1,cDNA,1,0,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4968...,RNA-Seq,"Feiyang,,Ma",Public on Jan 20 2022,aorta-gonad-mesonephros 5 week,GSE162950,SRA,9606,Illumina NovaSeq 6000,GSM4968832,USA,Homo sapiens,1,"610, Charles Young Dr East, TLSB 3000C",total RNA,time: 5wk,GSE162950,90095,Dec 09 2020,"For single cell RNA-sequencing, dissociated ce...",UCLA,Los Angeles,Jan 20 2022,GPL24676,aorta-gonad-mesonephros,transcriptomic,GSM4968832,Sequencing was performed on Illumina NovaSeq 6...,BioSample: https://www.ncbi.nlm.nih.gov/biosam...
2,cDNA,2,0,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4968...,RNA-Seq,"Feiyang,,Ma",Public on Jan 20 2022,aorta-gonad-mesonephros 5.5 week,GSE162950,SRA,9606,Illumina NovaSeq 6000,GSM4968833,USA,Homo sapiens,1,"610, Charles Young Dr East, TLSB 3000C",total RNA,time: 5wk,GSE162950,90095,Dec 09 2020,"For single cell RNA-sequencing, dissociated ce...",UCLA,Los Angeles,Jan 20 2022,GPL24676,aorta-gonad-mesonephros,transcriptomic,GSM4968833,Sequencing was performed on Illumina NovaSeq 6...,BioSample: https://www.ncbi.nlm.nih.gov/biosam...


In [186]:
geo['molecule_ch1'].value_counts()

molecule_ch1
polyA RNA      150
total RNA       96
protein         11
other           11
genomic DNA      4
Name: count, dtype: int64

In [187]:
geo['library_source'].value_counts()

library_source
transcriptomic                211
transcriptomic single cell     35
other                          22
genomic                         4
Name: count, dtype: int64

In [188]:
geo['library_strategy'].value_counts()

library_strategy
RNA-Seq     228
OTHER        40
ATAC-seq      4
Name: count, dtype: int64

In [189]:
geo['library_selection'].value_counts()

library_selection
cDNA     228
other     44
Name: count, dtype: int64

In [190]:
pd.crosstab(geo['molecule_ch1'],geo['library_strategy'])

library_strategy,ATAC-seq,OTHER,RNA-Seq
molecule_ch1,,,
genomic DNA,4,0,0
other,0,11,0
polyA RNA,0,7,143
protein,0,11,0
total RNA,0,11,85


In [191]:
# CITE, do not need it
pd.set_option('display.max_rows', 20)
geo.loc[geo['molecule_ch1']=='other','title']

261     TT-CITE-1-TCRab
262     TT-CITE-2-TCRab
263     TT-CITE-3-TCRab
264     TT-CITE-4-TCRab
265     TT-CITE-5-TCRab
266     TT-CITE-6-TCRab
267     TT-CITE-8-TCRab
268     TT-CITE-9-TCRab
269    TT-CITE-10-TCRab
270    TT-CITE-11-TCRab
271    TT-CITE-12-TCRab
Name: title, dtype: object

In [192]:
geo = geo.loc[(geo['library_strategy'] == 'RNA-Seq') | (geo['molecule_ch1'].isin(['polyA RNA','total RNA'])),]
pd.crosstab(geo['molecule_ch1'],geo['library_strategy'])

library_strategy,OTHER,RNA-Seq
molecule_ch1,,
polyA RNA,7,143
total RNA,11,85


In [193]:
geo=geo.loc[:,['dataset_acc','geo_accession','title']].drop_duplicates()
geo['library_type'] = 'NA'
geo

,dataset_acc,geo_accession,title,library_type
0,GSE162950,GSM4968831,aorta-gonad-mesonephros 4.5 week,NA
1,GSE162950,GSM4968832,aorta-gonad-mesonephros 5 week,NA
2,GSE162950,GSM4968833,aorta-gonad-mesonephros 5.5 week,NA
3,GSE162950,GSM4968834,aorta-gonad-mesonephros 6 week,NA
4,GSE162950,GSM4968835,cord 4.5 week,NA
...,...,...,...,...
245,GSE271304,GSM8374230,TT-CITE-8-GEX,NA
246,GSE271304,GSM8374231,TT-CITE-9-GEX,NA
247,GSE271304,GSM8374232,TT-CITE-10-GEX,NA
248,GSE271304,GSM8374233,TT-CITE-11-GEX,NA


## ENA

In [194]:
f = datasets.accession.str.startswith('PRJEB')
dss = []
for a in datasets.accession[f]:
    d = pd.read_csv("datasets_metadata/"+a+".csv")
    d['dataset_acc'] = a
    dss.append(d)
    # hooks for non-usual datasets
    if a =='PRJEB37165':
        d['sample_title'] = [x[0] for x in d['sample_title'].str.split('_')]

common_cols = list(set.intersection(*(set(df.columns) for df in dss)))

ena = pd.concat([df[common_cols] for df in dss], ignore_index=True)
ena = ena.loc[:,['dataset_acc','secondary_sample_accession','sample_title']].drop_duplicates()
ena['library_type'] = 'NA'
ena

,dataset_acc,secondary_sample_accession,sample_title,library_type
0,PRJEB37165,ERS4605095,4834STDY7002875,NA
1,PRJEB37165,ERS4605093,4834STDY7002881,NA
2,PRJEB37165,ERS4605082,4834STDY7002876,NA
3,PRJEB37165,ERS4605085,CZIKidney7587404,NA
4,PRJEB37165,ERS4605080,CZIKidney7632803,NA
...,...,...,...,...
63,PRJEB77091,ERS20592939,TA11486161,NA
64,PRJEB77091,ERS20592941,TA11486163,NA
65,PRJEB77091,ERS20592942,TA11486164,NA
66,PRJEB77091,ERS20592946,TA11556492,NA


## Internal

In [195]:
we_11pcv = pd.read_csv('datasets_metadata/whole_embryo_11pcw.csv')
we_11pcv['dataset_acc'] = 'whole_embryo_11pcw'
we_11pcv['library_type'] = 'NA'
we_11pcv = we_11pcv[['dataset_acc','Sanger ID','Sanger ID','library_type']].drop_duplicates()
we_11pcv

,dataset_acc,Sanger ID,Sanger ID,library_type
0,whole_embryo_11pcw,HDBI_wEMB14781754,HDBI_wEMB14781754,NA
1,whole_embryo_11pcw,HDBI_wEMB14781755,HDBI_wEMB14781755,NA
2,whole_embryo_11pcw,HDBI_wEMB14781756,HDBI_wEMB14781756,NA
3,whole_embryo_11pcw,HDBI_wEMB14781757,HDBI_wEMB14781757,NA
4,whole_embryo_11pcw,HDBI_wEMB14781758,HDBI_wEMB14781758,NA
...,...,...,...,...
43,whole_embryo_11pcw,HDBI_wEMB14781797,HDBI_wEMB14781797,NA
44,whole_embryo_11pcw,HDBI_wEMB14781798,HDBI_wEMB14781798,NA
45,whole_embryo_11pcw,HDBI_wEMB14781799,HDBI_wEMB14781799,NA
46,whole_embryo_11pcw,HDBI_wEMB14781800,HDBI_wEMB14781800,NA


In [196]:
internal = pd.read_csv('datasets_metadata/internal_datasets.csv')
internal['library_type'] = 'NA'
internal['dataset_acc'] = internal['dataset']
internal = internal[['dataset_acc','sample_title','sample_title','library_type']]
internal

,dataset_acc,sample_title,sample_title,library_type
0,Kanemaru_et_al_2024_bioRxiv,BHF_F_Hea10402917,BHF_F_Hea10402917,NA
1,Kanemaru_et_al_2024_bioRxiv,BHF_F_Hea10402918,BHF_F_Hea10402918,NA
2,Kanemaru_et_al_2024_bioRxiv,BHF_F_Hea11064670,BHF_F_Hea11064670,NA
3,Kanemaru_et_al_2024_bioRxiv,BHF_F_Hea11064671,BHF_F_Hea11064671,NA
4,Kanemaru_et_al_2024_bioRxiv,BHF_F_Hea11064672,BHF_F_Hea11064672,NA
...,...,...,...,...
38,Garcia_Alonso_et_al_2022_nature,HCA_F_GON10828900,HCA_F_GON10828900,NA
39,Garcia_Alonso_et_al_2022_nature,HCA_F_GON10828901,HCA_F_GON10828901,NA
40,Garcia_Alonso_et_al_2022_nature,HCA_F_GON10828902,HCA_F_GON10828902,NA
41,Garcia_Alonso_et_al_2022_nature,HCA_F_GON10941968,HCA_F_GON10941968,NA


## Merge

In [197]:
ae.columns = geo.columns = ena.columns = we_11pcv.columns = internal.columns  = ega.columns = ['dataset_acc','sample_acc','author_sample_id','library_type']

In [198]:
samples = pd.concat([ae,ega,geo,ena,we_11pcv,internal], ignore_index=True)
samples['institute'] = samples.dataset_acc.replace(dict(zip(datasets.accession,datasets.institute)))
samples['study'] = samples.dataset_acc.replace(dict(zip(datasets.accession,datasets.title)))
samples

,dataset_acc,sample_acc,author_sample_id,library_type,institute,study
0,E-MTAB-10187,ERS5890146,HT-185_Duodenum,10xV2,Roche,Yu_et_al_2020_Cell
1,E-MTAB-10187,ERS5890147,HT-188_Duodenum,10xV2,Roche,Yu_et_al_2020_Cell
2,E-MTAB-10187,ERS5890148,HT-228_Colon,10xV2,Roche,Yu_et_al_2020_Cell
3,E-MTAB-10187,ERS5890149,HT-228_Ileum,10xV2,Roche,Yu_et_al_2020_Cell
4,E-MTAB-10187,ERS5890150,HT-228_Liver,10xV2,Roche,Yu_et_al_2020_Cell
...,...,...,...,...,...,...
1672,Garcia_Alonso_et_al_2022_nature,HCA_F_GON10828900,HCA_F_GON10828900,NA,Wellcome_Sanger,Garcia_Alonso_et_al_2022_nature
1673,Garcia_Alonso_et_al_2022_nature,HCA_F_GON10828901,HCA_F_GON10828901,NA,Wellcome_Sanger,Garcia_Alonso_et_al_2022_nature
1674,Garcia_Alonso_et_al_2022_nature,HCA_F_GON10828902,HCA_F_GON10828902,NA,Wellcome_Sanger,Garcia_Alonso_et_al_2022_nature
1675,Garcia_Alonso_et_al_2022_nature,HCA_F_GON10941968,HCA_F_GON10941968,NA,Wellcome_Sanger,Garcia_Alonso_et_al_2022_nature


# Check completeness

In [199]:
set(datasets.accession) - set(samples.dataset_acc)

{'HRA002757', 'HRA005835'}

In [200]:
# tmp column to use to point to processed data on irods. 
# EGA samples are named by author_sample_id instead of sample_acc
samples['irods_name'] = samples['sample_acc']
f = samples['dataset_acc'].str.startswith('EGAD')
samples.loc[f,'irods_name'] = samples.loc[f,'author_sample_id']

# all sanger samples are named by Sanger id on irods
f = samples['institute'] == 'Wellcome_Sanger'
samples.loc[f,'irods_name'] = samples.loc[f,'author_sample_id']


In [201]:
# look for samples found in multiple submissions
t=samples.author_sample_id.value_counts()
t = [[x,', '.join(sorted(samples.study[samples.author_sample_id==x].tolist())),', '.join(sorted(samples.dataset_acc[samples.author_sample_id==x].tolist()))]  for x in t[t>1].index]
t=pd.DataFrame(t,columns=['author_sample_id','studies','datasets'])
t.sort_values('datasets')


,author_sample_id,studies,datasets
0,FCAImmP7862086,"Goh_yolk_sac_2023, Suo_et_al_2022_Science_Suo2022","E-MTAB-10552, E-MTAB-11343"
13,FCAImmP7862093,"Goh_yolk_sac_2023, Suo_et_al_2022_Science_Suo2022","E-MTAB-10552, E-MTAB-11343"
9,FCAImmP7862084,"Goh_yolk_sac_2023, Suo_et_al_2022_Science_Suo2022","E-MTAB-10552, E-MTAB-11343"
7,FCAImmP7862087,"Goh_yolk_sac_2023, Suo_et_al_2022_Science_Suo2022","E-MTAB-10552, E-MTAB-11343"
6,FCAImmP7862088,"Goh_yolk_sac_2023, Suo_et_al_2022_Science_Suo2022","E-MTAB-10552, E-MTAB-11343"
8,FCAImmP7862085,"Goh_yolk_sac_2023, Suo_et_al_2022_Science_Suo2022","E-MTAB-10552, E-MTAB-11343"
4,FCAImmP7862090,"Goh_yolk_sac_2023, Suo_et_al_2022_Science_Suo2022","E-MTAB-10552, E-MTAB-11343"
3,FCAImmP7862091,"Goh_yolk_sac_2023, Suo_et_al_2022_Science_Suo2022","E-MTAB-10552, E-MTAB-11343"
2,FCAImmP7862092,"Goh_yolk_sac_2023, Suo_et_al_2022_Science_Suo2022","E-MTAB-10552, E-MTAB-11343"
5,FCAImmP7862089,"Goh_yolk_sac_2023, Suo_et_al_2022_Science_Suo2022","E-MTAB-10552, E-MTAB-11343"


In [202]:
# keep first dataset if sample found in two datasets
tmp = samples[samples.dataset_acc==datasets.accession[0]]
for a in datasets.accession[1:]:
    tmp = pd.concat([tmp,samples[(samples.dataset_acc==a) & ~(samples.irods_name.isin(tmp.irods_name))]])
samples = tmp
samples.index = samples.sample_acc

In [203]:
# samples from HDCA v1
samples_v1 = pd.read_csv('libraries_v1.csv')
samples_v1

,institute,study,library_id_fixed,n_cells,library_id_is_sanger,n_unique_bs,dataset_acc,library_acc,Unnamed: 8,comment
0,Karolinska,Braun_et_al._2022_bioRxiv,10X101_1,6610,False,6610,EGAD00001006049,10X101_1,NaN,NaN
1,Karolinska,Braun_et_al._2022_bioRxiv,10X101_2,7579,False,7579,EGAD00001006049,10X101_2,NaN,NaN
2,Karolinska,Braun_et_al._2022_bioRxiv,10X101_3,7635,False,7635,EGAD00001006049,10X101_3,NaN,NaN
3,Karolinska,Braun_et_al._2022_bioRxiv,10X101_4,6970,False,6970,EGAD00001006049,10X101_4,NaN,NaN
4,Karolinska,Braun_et_al._2022_bioRxiv,10X101_5,6135,False,6135,EGAD00001006049,10X101_5,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
975,Wellcome_Sanger,Zhang_et_al_2020_bioRxiv,WSSS_THYst9384954,3557,True,3557,Sanger,WSSS_THYst9384954,NaN,NaN
976,Wellcome_Sanger,Zhang_et_al_2020_bioRxiv,WSSS_THYst9384955,3917,True,3917,Sanger,WSSS_THYst9384955,NaN,NaN
977,Wellcome_Sanger,Zhang_et_al_2020_bioRxiv,WSSS_THYst9384956,4862,True,4862,Sanger,WSSS_THYst9384956,NaN,NaN
978,Wellcome_Sanger,Zhang_et_al_2020_bioRxiv,WSSS_THYst9384957,3624,True,3624,Sanger,WSSS_THYst9384957,NaN,NaN


In [204]:
samples_v1[~samples_v1.library_acc.isin(samples.irods_name)]

,institute,study,library_id_fixed,n_cells,library_id_is_sanger,n_unique_bs,dataset_acc,library_acc,Unnamed: 8,comment
111,Karolinska,Braun_et_al._2022_bioRxiv,10X147_1,6127,False,6127,EGAD50000001295,10X147_1_ABC_1,NaN,NaN
193,Karolinska,Braun_et_al._2022_bioRxiv,10X186_1,3521,False,3521,EGAD50000001295,10X186_1_ABC_1,NaN,NaN
195,Karolinska,Braun_et_al._2022_bioRxiv,10X186_4,5708,False,5708,EGAD50000001295,10X186_4_ABC_1,NaN,NaN
196,Karolinska,Braun_et_al._2022_bioRxiv,10X187_1,2692,False,2692,EGAD50000001295,10X187_1_ABC_1,NaN,NaN
197,Karolinska,Braun_et_al._2022_bioRxiv,10X187_2,2478,False,2478,EGAD50000001295,10X187_2_ABC_1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
321,Karolinska,Braun_et_al._2022_bioRxiv,10X302_4,2485,False,2485,EGAD50000001295,10X302_4_AB_1,NaN,NaN
322,Karolinska,Braun_et_al._2022_bioRxiv,10X302_5,2123,False,2123,EGAD50000001295,10X302_5_AB_1,NaN,NaN
323,Karolinska,Braun_et_al._2022_bioRxiv,10X302_6,2324,False,2324,EGAD50000001295,10X302_6_AB_1,NaN,NaN
324,Karolinska,Braun_et_al._2022_bioRxiv,10X302_7,1839,False,1839,EGAD50000001295,10X302_7_AB_1,NaN,NaN


In [205]:
# not sure where the _ABC_1/AB_ in Braun are coming from...

In [206]:
samples_v1_braun = samples_v1[samples_v1.study == 'Braun_et_al._2022_bioRxiv']
samples_v1_braun = dict(zip(samples_v1_braun.library_id_fixed,samples_v1_braun.library_acc))
samples.irods_name = samples.irods_name.replace(samples_v1_braun)

In [207]:
samples_v1.loc[~samples_v1.library_acc.isin(samples.irods_name),'study'].value_counts()

Series([], Name: count, dtype: int64)

In [208]:
# He_et_al_2022_Cell was droped, that's is fine

In [209]:
samples_v1.loc[(samples_v1.study=='Suo_et_al_2022_Science_Elmentaite2020')]

,institute,study,library_id_fixed,n_cells,library_id_is_sanger,n_unique_bs,dataset_acc,library_acc,Unnamed: 8,comment
563,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,FCA_gut8015057,760,True,760,Sanger,FCA_gut8015057,NaN,NaN
564,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,FCA_gut8015058,1827,True,1827,Sanger,FCA_gut8015058,NaN,NaN
565,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,FCA_gut8015059,1524,True,1524,Sanger,FCA_gut8015059,NaN,NaN
566,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,FCA_gut8015060,1893,True,1893,Sanger,FCA_gut8015060,NaN,NaN
567,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,FCA_gut8015061,3157,True,3157,Sanger,FCA_gut8015061,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
579,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,Human_colon_16S8159186,2882,True,2882,Sanger,Human_colon_16S8159186,NaN,NaN
580,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,Human_colon_16S8159187,10722,True,10722,Sanger,Human_colon_16S8159187,NaN,NaN
581,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,Human_colon_16S8159188,8917,True,8917,Sanger,Human_colon_16S8159188,NaN,NaN
582,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,Human_colon_16S8159189,3294,True,3294,Sanger,Human_colon_16S8159189,NaN,NaN


In [210]:
samples[samples.study=='Suo_et_al_2022_Science_Elmentaite2020']

,dataset_acc,sample_acc,author_sample_id,library_type,institute,study,irods_name
sample_acc,,,,,,,
ERS23679606,E-MTAB-9536,ERS23679606,FCA_gut8015057,10x 5' v2,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,FCA_gut8015057
ERS23679607,E-MTAB-9536,ERS23679607,FCA_gut8015058,10x 5' v2,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,FCA_gut8015058
ERS23679608,E-MTAB-9536,ERS23679608,FCA_gut8015059,10x 5' v2,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,FCA_gut8015059
ERS23679609,E-MTAB-9536,ERS23679609,FCA_gut8015060,10x 5' v2,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,FCA_gut8015060
ERS23679610,E-MTAB-9536,ERS23679610,FCA_gut8015061,10x 5' v2,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,FCA_gut8015061
...,...,...,...,...,...,...,...
ERS23679627,E-MTAB-9536,ERS23679627,Human_colon_16S8159186,10x 5' v2,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,Human_colon_16S8159186
ERS23679628,E-MTAB-9536,ERS23679628,Human_colon_16S8159187,10x 5' v2,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,Human_colon_16S8159187
ERS23679629,E-MTAB-9536,ERS23679629,Human_colon_16S8159188,10x 5' v2,Wellcome_Sanger,Suo_et_al_2022_Science_Elmentaite2020,Human_colon_16S8159188


In [211]:
samples[samples.irods_name=='4834STDY7002881']

,dataset_acc,sample_acc,author_sample_id,library_type,institute,study,irods_name
sample_acc,,,,,,,
ERS4605093,PRJEB37165,ERS4605093,4834STDY7002881,NA,Wellcome_Sanger,Suo_et_al_2022_Science_Stewart2019,4834STDY7002881


In [212]:
samples['HDCA_version'] = 'v2'
samples.loc[samples.irods_name.isin(samples_v1.library_acc),'HDCA_version'] = 'v1'
samples['HDCA_version'].value_counts()

HDCA_version
v1    980
v2    679
Name: count, dtype: int64

In [213]:
samples

,dataset_acc,sample_acc,author_sample_id,library_type,institute,study,irods_name,HDCA_version
sample_acc,,,,,,,,
GSM4968831,GSE162950,GSM4968831,aorta-gonad-mesonephros 4.5 week,NA,Broad_Stem_Cell_Research_centre,Calvanese_et_al_2022_nature,GSM4968831,v1
GSM4968832,GSE162950,GSM4968832,aorta-gonad-mesonephros 5 week,NA,Broad_Stem_Cell_Research_centre,Calvanese_et_al_2022_nature,GSM4968832,v1
GSM4968833,GSE162950,GSM4968833,aorta-gonad-mesonephros 5.5 week,NA,Broad_Stem_Cell_Research_centre,Calvanese_et_al_2022_nature,GSM4968833,v1
GSM4968834,GSE162950,GSM4968834,aorta-gonad-mesonephros 6 week,NA,Broad_Stem_Cell_Research_centre,Calvanese_et_al_2022_nature,GSM4968834,v1
GSM4968835,GSE162950,GSM4968835,cord 4.5 week,NA,Broad_Stem_Cell_Research_centre,Calvanese_et_al_2022_nature,GSM4968835,v2
...,...,...,...,...,...,...,...,...
ERS4228604,E-MTAB-8581,ERS4228604,WSSS8062675,10xV2,Wellcome_Sanger,Yayon_thymus_2024,WSSS8062675,v2
ERS4228662,E-MTAB-8581,ERS4228662,WSSS8084742,10xV3,Wellcome_Sanger,Yayon_thymus_2024,WSSS8084742,v2
ERS4228663,E-MTAB-8581,ERS4228663,WSSS8084743,10xV3,Wellcome_Sanger,Yayon_thymus_2024,WSSS8084743,v2


# Add extra info for sanger samples

In [218]:
import importlib
importlib.reload(utils.fetch_dbs)
with open('utils/secrets.json', 'r') as f:
    secrets = json.load(f)

In [226]:
utils.fetch_dbs.fetch_mlw_query(secrets['mlw'],'SELECT * from sample WHERE supplier_name="T06_TH_TOT_VDJT_3"')

,id_sample_tmp,id_lims,uuid_sample_lims,id_sample_lims,last_updated,recorded_at,deleted_at,created,name,reference_genome,organism,accession_number,common_name,description,taxon_id,father,mother,replicate,ethnicity,gender,cohort,country_of_origin,geographical_region,sanger_sample_id,control,supplier_name,public_name,sample_visibility,strain,consent_withdrawn,donor_id,phenotype,developmental_stage,control_type,sibling,is_resubmitted,date_of_sample_collection,date_of_sample_extraction,extraction_method,purified,purification_method,customer_measured_concentration,concentration_determined_by,sample_type,storage_conditions,genotype,age,cell_type,disease_state,compound,dose,immunoprecipitate,growth_condition,organism_part,time_point,disease,subject,treatment,date_of_consent_withdrawn,marked_as_consent_withdrawn_by,customer_measured_volume,gc_content,dna_source,priority_level


In [217]:
samples[samples.dataset_acc=='E-MTAB-11278']

,dataset_acc,sample_acc,author_sample_id,library_type,institute,study,irods_name,HDCA_version
sample_acc,,,,,,,,
ERS20107579,E-MTAB-11278,ERS20107579,5478STDY7698210,10x 3' v2,Wellcome_Sanger,He_et_al_2022_Cell,5478STDY7698210,v2
ERS20107580,E-MTAB-11278,ERS20107580,5698STDY7839908,10x 3' v2,Wellcome_Sanger,He_et_al_2022_Cell,5698STDY7839908,v2
ERS20107581,E-MTAB-11278,ERS20107581,5698STDY7839910,10x 3' v3,Wellcome_Sanger,He_et_al_2022_Cell,5698STDY7839910,v2
ERS20107582,E-MTAB-11278,ERS20107582,5698STDY7839918,10x 5' v1,Wellcome_Sanger,He_et_al_2022_Cell,5698STDY7839918,v2
ERS20107584,E-MTAB-11278,ERS20107584,5891STDY8062349,10x 5' v1,Wellcome_Sanger,He_et_al_2022_Cell,5891STDY8062349,v1
...,...,...,...,...,...,...,...,...
ERS20107652,E-MTAB-11278,ERS20107652,WSSS_F_LNG8713189,10x 5' v1,Wellcome_Sanger,He_et_al_2022_Cell,WSSS_F_LNG8713189,v1
ERS20107653,E-MTAB-11278,ERS20107653,WSSS_F_LNG8713190,10x 5' v1,Wellcome_Sanger,He_et_al_2022_Cell,WSSS_F_LNG8713190,v1
ERS20107654,E-MTAB-11278,ERS20107654,WSSS_F_LNG8713191,10x 5' v1,Wellcome_Sanger,He_et_al_2022_Cell,WSSS_F_LNG8713191,v1


In [220]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
sids = samples.loc[samples.institute=='Wellcome_Sanger','author_sample_id'].tolist()
sanger_sample_info = utils.fetch_dbs.get_sample_info_from_mlw(secrets['mlw'],sids)
sanger_sample_info

,sample_name,sqsc_sample_id,supplier_name,donor_id,study_name,sqsc_study_id,pipeline_id_lims
0,FCA_GND10375779,5854186,Hrv86-GON-0-SC-1,FCA_GND10375779,Developmental Gonads,5888,Chromium single cell 5 prime
0,FCA_GND10375780,5854187,Hrv86-GON-0-SC-2,FCA_GND10375780,Developmental Gonads,5888,Chromium single cell 5 prime
0,FCA_GND8047885,4171842,F81-GON-0-SC-F-45N-2,FCA_GND8047885,Developmental Gonads,5888,Chromium single cell
0,FCA_GND8103050,4205389,F83-GON-0-SC-F-45N-1,FCA_GND8103050,Developmental Gonads,5888,Chromium single cell 5 prime
0,FCA_GND8103053,4205392,F84-GON-0-SC-F-45N-3,FCA_GND8103053,Developmental Gonads,5888,Chromium single cell 5 prime
...,...,...,...,...,...,...,...
0,WSSS8062675,4179161,A43_TH_CD45N_5GEX_2,WSSS8062675,WSSS_Research_Study_HCA,5857,Chromium single cell
0,WSSS8084742,4180204,C40_TH_TOT_1,WSSS8084742,WSSS_Research_Study_HCA,5857,Chromium single cell
0,WSSS8084743,4180205,C40_TH_TOT_2,WSSS8084743,WSSS_Research_Study_HCA,5857,Chromium single cell
0,WSSS8084744,4180206,C41_TH_TOT_1,WSSS8084744,WSSS_Research_Study_HCA,5857,Chromium single cell


In [227]:
samples["found_internal"] = False
samples.loc[samples.author_sample_id.isin(sanger_sample_info.sample_name),"found_internal"] = True
sanger_not_sanger = samples[~samples["found_internal"] & (samples.institute=='Wellcome_Sanger')]
sanger_not_sanger.study.value_counts()

study
Yayon_thymus_2024                     28
He_et_al_2022_Cell                    10
Suo_et_al_2022_Science_Jardine2021     5
Name: count, dtype: int64

In [228]:
sanger_not_sanger[sanger_not_sanger.study=='Suo_et_al_2022_Science_Jardine2021']

,dataset_acc,sample_acc,author_sample_id,library_type,institute,study,irods_name,HDCA_version,found_internal
sample_acc,,,,,,,,,
ERS4853347,E-MTAB-9389,ERS4853347,DSOX19_1,10xV2,Wellcome_Sanger,Suo_et_al_2022_Science_Jardine2021,DSOX19_1,v2,False
ERS4853348,E-MTAB-9389,ERS4853348,DSOX19_2,10xV2,Wellcome_Sanger,Suo_et_al_2022_Science_Jardine2021,DSOX19_2,v2,False
ERS4853349,E-MTAB-9389,ERS4853349,DSOX4,10xV2,Wellcome_Sanger,Suo_et_al_2022_Science_Jardine2021,DSOX4,v2,False
ERS4853350,E-MTAB-9389,ERS4853350,DSOX7,10xV2,Wellcome_Sanger,Suo_et_al_2022_Science_Jardine2021,DSOX7,v2,False
ERS4853351,E-MTAB-9389,ERS4853351,DSOX8,10xV2,Wellcome_Sanger,Suo_et_al_2022_Science_Jardine2021,DSOX8,v2,False


In [231]:
# what are we doing with
# 1) not Sanger samples in Sanger submissions
# 2) not HDCA samples in otherwise HDCA submissions: 
# 2.1) skip for v1 datasets?